# Mesa de ayuda interna de TI

Prototipo del trabajo integrador del curso **Ciberseguridad de Inteligencia Artificial** ·
Maestría en Ciencia de Datos, UNI · Grupo **Ciber2026**.

> **Producto ficticio, datos sintéticos.** La organización, los documentos, los usuarios
> y las credenciales son inventados. No corresponden a ningún sistema real.

Funciona **sin instalar nada y sin llave de API**. Si además tienes una llave de DeepSeek
cargada en los secretos de Colab, el notebook la detecta solo y responde con ese modelo.

Ejecuta las celdas en orden con `Shift + Enter`.

## 1. Traer el proyecto

Clona el repositorio dentro de la máquina temporal de Colab. Vuelve a ejecutarse sin
problema si la sesión se reinicia.

In [ ]:
import os, shutil, subprocess, sys

REPO = "https://github.com/NormanUNI/grupo_2-mesa-ayuda-ti.git"
CARPETA = "grupo_2-mesa-ayuda-ti"

if os.path.isdir(CARPETA):
    shutil.rmtree(CARPETA)
subprocess.run(["git", "clone", "--depth", "1", REPO, CARPETA], check=True)
os.chdir(CARPETA)
sys.path.insert(0, os.getcwd())

print("Listo. Carpeta de trabajo:", os.getcwd())

## 2. Cargar el asistente

Esta celda elige el modelo y no hay que tocar nada más después:

| Si en los secretos de Colab hay… | El asistente responde con… |
| --- | --- |
| un secreto `DEEPSEEK_API_KEY` con *Acceso del notebook* activado | **DeepSeek** |
| nada | **`eco`**, un modelo local de prueba, sin red ni llave |

`eco` no es un modelo de lenguaje: arma la respuesta a partir de los documentos
recuperados. Sirve para recorrer el producto y ver cómo funciona, **no para medir**.
Una cifra obtenida contra `eco` no dice nada sobre el comportamiento de un modelo real,
por eso la celda imprime siempre cuál quedó activo.

Para cambiar de `eco` a DeepSeek: icono de la llave 🔑 en la barra izquierda →
**+ Agregar secreto** → nombre exacto `DEEPSEEK_API_KEY` → activa *Acceso del notebook*
→ vuelve a ejecutar esta celda. La llave **nunca va escrita en el notebook**: este
repositorio es público y cualquiera que lo abriera la vería.

`Controles()` sin argumentos deja los tres interruptores **apagados**. Esa es la línea
base del producto, y es una decisión deliberada de la arquitectura, no un descuido.

In [ ]:
import os

LLAVE = None
try:
    from google.colab import userdata
    LLAVE = userdata.get("DEEPSEEK_API_KEY")
except ImportError:
    LLAVE = os.environ.get("DEEPSEEK_API_KEY")   # fuera de Colab
except Exception:
    pass   # no existe el secreto, o no se le dio acceso a este notebook

if LLAVE:
    import subprocess, sys
    # DeepSeek expone un formato compatible con OpenAI, asi que usa ese paquete.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai"], check=True)
    os.environ["DEEPSEEK_API_KEY"] = LLAVE
    PROVEEDOR = "deepseek"
else:
    PROVEEDOR = "eco"

# config lee la llave del entorno al importarse. El reload hace que esta celda
# funcione tambien cuando ya se habia ejecutado antes, sin reiniciar la sesion.
import importlib
from src import config, modelo
importlib.reload(config)
importlib.reload(modelo)

from src.asistente import Asistente, Controles
from src.registro import Registro, leer
from src import herramientas

registro = Registro(f"colab-{PROVEEDOR}")
asistente = Asistente(registro=registro, controles=Controles(), proveedor=PROVEEDOR)

print(f"modelo:  {asistente.modelo.proveedor}/{asistente.modelo.nombre}")
if PROVEEDOR == "eco":
    print("         modo local de prueba: sirve para recorrer el producto, no para medir")
print(f"corpus:  {len(asistente.documentos)} documentos sinteticos")
print(f"usuarios:")
for uid, u in asistente.usuarios.items():
    print(f"  {uid}  {u['nombre']:<16} rol {u['rol']:<14} ve {u['clasificaciones_permitidas']}")

## 3. Una función para preguntar

Envuelve al asistente para que cada consulta muestre lo mismo: la respuesta, qué
documentos se recuperaron con su clasificación, y qué herramientas se invocaron.

In [ ]:
def preguntar(consulta, usuario="u-001", controles=False):
    """Envia una consulta al asistente y muestra el resultado."""
    asistente.controles = Controles(controles, controles, controles)
    r = asistente.responder(consulta, id_usuario=usuario)

    estado = "ACTIVADOS" if controles else "apagados (linea base)"
    print(f"[{usuario}] {consulta}")
    print(f"controles: {estado}")
    print("-" * 72)
    print(r.texto)
    print("-" * 72)

    if r.documentos:
        for d in r.documentos:
            print(f"  doc  {d['id']}  {d['clasificacion']:<12} {d['titulo']}")
    else:
        print("  doc  ninguno recuperado")

    for inv in r.herramientas:
        marca = "ejecutada" if inv["permitida"] else "BLOQUEADA"
        print(f"  tool {inv['herramienta']}: {marca}")
    print()
    return r

## 4. Consultar desde un cuadro de texto

Ejecuta la celda una sola vez. Aparece un formulario: eliges el usuario, escribes
la consulta y pulsas **Preguntar**. No hace falta volver a tocar código — puedes
preguntar las veces que quieras sin reejecutar nada.

La casilla **Activar controles** enciende los tres interruptores de seguridad a la vez:
mínimo privilegio en la recuperación, aislamiento de memoria y lista blanca de
herramientas. Medir qué hace cada uno es trabajo de la sesión 7; aquí solo quedan
expuestos.

In [ ]:
import io
import contextlib
import ipywidgets as widgets
from IPython.display import display, clear_output

_opciones = [
    (f"{uid} · {u['nombre']} · {u['rol']}", uid)
    for uid, u in asistente.usuarios.items()
]

_usuario = widgets.Dropdown(
    options=_opciones, description="Usuario:",
    layout=widgets.Layout(width="440px"),
)
_controles = widgets.Checkbox(
    value=False, description="Activar controles de seguridad", indent=False,
    layout=widgets.Layout(width="440px"),
)
_consulta = widgets.Textarea(
    placeholder="Escribe tu consulta y pulsa Preguntar...",
    layout=widgets.Layout(width="660px", height="80px"),
)
_boton = widgets.Button(description="Preguntar", button_style="primary")
_limpiar = widgets.Button(description="Limpiar")
_salida = widgets.Output()


def _al_preguntar(_):
    """Consulta al asistente y muestra el resultado en el area de salida.

    preguntar() escribe en stdout, asi que se captura para poder mostrar
    "Consultando..." y reemplazarlo despues. Se invoca UNA sola vez: llamarla
    dos veces duplicaria el costo de la API y las entradas del registro.
    """
    texto = _consulta.value.strip()
    with _salida:
        clear_output(wait=True)
        if not texto:
            print("Escribe una consulta primero.")
            return
        print("Consultando...")

    buffer = io.StringIO()
    try:
        with contextlib.redirect_stdout(buffer):
            preguntar(texto, usuario=_usuario.value, controles=_controles.value)
        resultado = buffer.getvalue()
    except Exception as error:
        resultado = f"Error: {type(error).__name__}: {error}"

    with _salida:
        clear_output(wait=True)
        print(resultado)


def _al_limpiar(_):
    _consulta.value = ""
    with _salida:
        clear_output()


_boton.on_click(_al_preguntar)
_limpiar.on_click(_al_limpiar)

display(
    widgets.VBox([
        _usuario,
        _controles,
        _consulta,
        widgets.HBox([_boton, _limpiar]),
        _salida,
    ])
)

### Consultas para probar

Cópialas en el cuadro de arriba.

| Consulta | Qué muestra |
| --- | --- |
| Cual es el plazo para reportar un incidente de seguridad? | Respuesta directa, citando el documento |
| Cuanto espacio tiene mi buzon de correo? | Consulta cotidiana |
| Cada cuanto vence mi contrasena de red? | Otra consulta de uso diario |
| Que significa el codigo de error 812 en la VPN? | Elige el usuario `u-002`: un colaborador común no ve ese documento |
| Abrir un ticket urgente porque no puedo conectarme a la VPN | El agente crea un ticket |
| Consultar el estado del ticket TCK-0001 | El agente lee el ticket que acaba de crear |

## 5. Consultas de ejemplo en código

Edita el texto y vuelve a ejecutar la celda las veces que quieras.

In [ ]:
preguntar("Cuanto espacio tiene mi buzon de correo?")
preguntar("Cada cuanto vence mi contrasena de red?")
preguntar("Que hago si mi impresion sale en blanco?")

Como usuario de mesa de ayuda (`u-002`), que sí puede ver los documentos `interno`:

In [ ]:
preguntar("Que significa el codigo de error 812 en la VPN?", usuario="u-002")
preguntar("Cuando se escala un ticket de nivel 1 a nivel 2?", usuario="u-002")

## 6. El agente ejecutando herramientas

El planificador reconoce frases específicas. `restablecer_acceso` es la única acción
marcada como irreversible.

In [ ]:
herramientas.reiniciar()

preguntar("Abrir un ticket urgente porque no puedo conectarme a la VPN", usuario="u-002")
preguntar("Consultar el estado del ticket TCK-0001", usuario="u-002")

## 7. El registro de eventos

Cada consulta deja una línea JSON por evento: consulta recibida, memoria recuperada,
control evaluado, documentos recuperados, herramienta invocada, respuesta entregada
y memoria escrita.

Este registro es lo que convierte al prototipo en algo medible. Sin él no hay forma
de sostener con evidencia qué pasó en cada consulta.

In [ ]:
eventos = leer(registro.ruta)
print(f"{len(eventos)} eventos en {registro.ruta}\n")

for e in eventos[-8:]:
    print(f"{e['marca_tiempo'][11:19]}  {e['tipo']}")

Un evento completo, para ver todos sus campos:

In [ ]:
import json

recuperados = [e for e in eventos if e["tipo"] == "documentos_recuperados"]
print(json.dumps(recuperados[-1], ensure_ascii=False, indent=2))

---

Repositorio: https://github.com/NormanUNI/grupo_2-mesa-ayuda-ti ·
Documentación: https://normanuni.github.io/grupo_2-mesa-ayuda-ti/